# Version 5: CNN-BiLSTM-Attention-CRF

Advanced architecture targeting 0.60+:
1. **1D CNN** for local context extraction
2. **3-layer Bi-LSTM** with residual connections
3. **Self-Attention** mechanism
4. **CRF Layer** for structured prediction
5. **Focal Loss** for hard example mining

In [ ]:
!pip install pytorch-crf -q

In [ ]:
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split
from torch.nn.utils.rnn import pad_sequence, pack_padded_sequence, pad_packed_sequence
from torchcrf import CRF
import pandas as pd
import numpy as np
from sklearn.metrics import f1_score
from sklearn.utils.class_weight import compute_class_weight
from tqdm.auto import tqdm

class Config:
    seed = 42
    device = torch.device('cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu')
    batch_size = 16  # Smaller for larger model
    embed_dim = 128
    cnn_channels = 256
    hidden_dim = 512  # Larger
    num_layers = 3    # Deeper
    dropout = 0.3
    learning_rate = 0.001
    epochs = 30
    focal_gamma = 2.0
    
    train_path = "./train.csv"
    test_path = "./test.csv"
    
    trackio_project = "25-t3-nppe2"
    trackio_space_id = "manyakaistha/dlgenai-nppe"
    trackio_group = "cnn_bilstm_crf_v5"

cfg = Config()
print(f"Device: {cfg.device}")

def set_seed(seed):
    torch.manual_seed(seed)
    np.random.seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
set_seed(cfg.seed)

os.environ["HUGGINGFACEHUB_API_TOKEN"] = "hf_placeholder_token_for_reproducibility"
os.environ["TRACKIO_SPACE_ID"] = cfg.trackio_space_id

def maybe_init_trackio():
    try:
        import trackio
        token = os.environ.get("HUGGINGFACEHUB_API_TOKEN") or os.environ.get("HF_TOKEN")
        if cfg.trackio_space_id and token:
            trackio.init(project=cfg.trackio_project, space_id=cfg.trackio_space_id, group=cfg.trackio_group)
            trackio.log({"status/started": 1})
            print("✓ TrackIO initialized")
            return trackio
    except Exception as e:
        print(f"TrackIO init failed: {e}")
    return None

trk = maybe_init_trackio()

## Data Preprocessing

In [ ]:
PAD_TOKEN = 0

AA_CHARS = ['<PAD>', '<UNK>', '*', 'A', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'K', 'L', 'M', 'N', 'P', 'Q', 'R', 'S', 'T', 'V', 'W', 'Y']
AA_TO_ID = {c: i for i, c in enumerate(AA_CHARS)}

Q8_CHARS = ['<PAD>'] + sorted(['B', 'C', 'E', 'G', 'H', 'I', 'S', 'T'])
Q8_TO_ID = {c: i for i, c in enumerate(Q8_CHARS)}
ID_TO_Q8 = {i: c for c, i in Q8_TO_ID.items()}

Q3_CHARS = ['<PAD>'] + sorted(['C', 'E', 'H'])
Q3_TO_ID = {c: i for i, c in enumerate(Q3_CHARS)}
ID_TO_Q3 = {i: c for c, i in Q3_TO_ID.items()}

class ProteinDataset(Dataset):
    def __init__(self, csv_path, mode='train'):
        self.data = pd.read_csv(csv_path)
        self.mode = mode
    def __len__(self): return len(self.data)
    def __getitem__(self, idx):
        row = self.data.iloc[idx]
        seq_ids = [AA_TO_ID.get(c, AA_TO_ID['<UNK>']) for c in row['seq']]
        item = {'seq': torch.tensor(seq_ids, dtype=torch.long), 'id': row['id']}
        if self.mode == 'train':
            item['q8'] = torch.tensor([Q8_TO_ID.get(c, 0) for c in row['sst8']], dtype=torch.long)
            item['q3'] = torch.tensor([Q3_TO_ID.get(c, 0) for c in row['sst3']], dtype=torch.long)
        return item

def collate_fn(batch):
    batch.sort(key=lambda x: len(x['seq']), reverse=True)
    seqs = [item['seq'] for item in batch]
    lengths = torch.tensor([len(s) for s in seqs], dtype=torch.long)
    seqs_padded = pad_sequence(seqs, batch_first=True, padding_value=PAD_TOKEN)
    out = {'seq': seqs_padded, 'ids': [item['id'] for item in batch], 'lengths': lengths}
    if 'q8' in batch[0]:
        out['q8'] = pad_sequence([item['q8'] for item in batch], batch_first=True, padding_value=PAD_TOKEN)
        out['q3'] = pad_sequence([item['q3'] for item in batch], batch_first=True, padding_value=PAD_TOKEN)
    return out

train_ds = ProteinDataset(cfg.train_path, mode='train')
val_size = int(0.1 * len(train_ds))
train_subset, val_subset = random_split(train_ds, [len(train_ds) - val_size, val_size], generator=torch.Generator().manual_seed(cfg.seed))
train_loader = DataLoader(train_subset, batch_size=cfg.batch_size, shuffle=True, collate_fn=collate_fn)
val_loader = DataLoader(val_subset, batch_size=cfg.batch_size, shuffle=False, collate_fn=collate_fn)
print(f"Train: {len(train_subset)}, Val: {len(val_subset)}")

## Focal Loss

In [ ]:
class FocalLoss(nn.Module):
    """Focal Loss for class imbalance. Focuses on hard examples."""
    def __init__(self, gamma=2.0, weight=None, ignore_index=-100):
        super().__init__()
        self.gamma = gamma
        self.weight = weight
        self.ignore_index = ignore_index
    
    def forward(self, logits, targets):
        # logits: [B*L, C], targets: [B*L]
        ce_loss = F.cross_entropy(logits, targets, weight=self.weight, ignore_index=self.ignore_index, reduction='none')
        p_t = torch.exp(-ce_loss)
        focal_loss = ((1 - p_t) ** self.gamma) * ce_loss
        return focal_loss.mean()

## Model: CNN-BiLSTM-Attention-CRF

In [ ]:
class SelfAttention(nn.Module):
    """Simple self-attention for sequence weighting."""
    def __init__(self, hidden_dim):
        super().__init__()
        self.attention = nn.Linear(hidden_dim, 1)
    
    def forward(self, x, mask=None):
        # x: [B, L, H]
        scores = self.attention(x).squeeze(-1)  # [B, L]
        if mask is not None:
            scores = scores.masked_fill(~mask, float('-inf'))
        weights = F.softmax(scores, dim=-1).unsqueeze(-1)  # [B, L, 1]
        # Weighted combination added to original
        context = (x * weights).sum(dim=1, keepdim=True)  # [B, 1, H]
        return x + context.expand_as(x)  # Residual attention

class CNN_BiLSTM_CRF(nn.Module):
    def __init__(self, vocab_size, embed_dim, cnn_channels, hidden_dim, num_layers, num_q8, num_q3, dropout=0.3):
        super().__init__()
        self.num_q8 = num_q8
        self.num_q3 = num_q3
        
        # Embedding
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=PAD_TOKEN)
        
        # 1D CNN Block
        self.conv3 = nn.Conv1d(embed_dim, cnn_channels // 2, kernel_size=3, padding=1)
        self.conv5 = nn.Conv1d(embed_dim, cnn_channels // 2, kernel_size=5, padding=2)
        self.cnn_norm = nn.LayerNorm(cnn_channels)
        self.cnn_dropout = nn.Dropout(dropout)
        
        # BiLSTM (stacked manually for residual)
        self.lstm_layers = nn.ModuleList()
        self.lstm_norms = nn.ModuleList()
        
        # First LSTM layer
        self.lstm_layers.append(nn.LSTM(cnn_channels, hidden_dim, 1, batch_first=True, bidirectional=True))
        self.lstm_norms.append(nn.LayerNorm(hidden_dim * 2))
        
        # Subsequent LSTM layers with residual
        for _ in range(num_layers - 1):
            self.lstm_layers.append(nn.LSTM(hidden_dim * 2, hidden_dim, 1, batch_first=True, bidirectional=True))
            self.lstm_norms.append(nn.LayerNorm(hidden_dim * 2))
        
        self.lstm_dropout = nn.Dropout(dropout)
        
        # Self-Attention
        self.attention = SelfAttention(hidden_dim * 2)
        
        # Emission heads
        self.q8_emission = nn.Linear(hidden_dim * 2, num_q8)
        self.q3_emission = nn.Linear(hidden_dim * 2, num_q3)
        
        # CRF layers
        self.crf_q8 = CRF(num_q8, batch_first=True)
        self.crf_q3 = CRF(num_q3, batch_first=True)
        
        self._init_weights()
    
    def _init_weights(self):
        nn.init.xavier_uniform_(self.q8_emission.weight)
        nn.init.xavier_uniform_(self.q3_emission.weight)
        nn.init.xavier_uniform_(self.conv3.weight)
        nn.init.xavier_uniform_(self.conv5.weight)
    
    def _get_mask(self, lengths, max_len):
        # Create boolean mask [B, L]
        batch_size = lengths.size(0)
        mask = torch.arange(max_len, device=lengths.device).expand(batch_size, max_len) < lengths.unsqueeze(1)
        return mask
    
    def forward(self, x, lengths, q8_tags=None, q3_tags=None):
        # x: [B, L]
        B, L = x.size()
        mask = self._get_mask(lengths, L)
        
        # Embedding
        embed = self.embedding(x)  # [B, L, E]
        
        # CNN: [B, L, E] -> [B, E, L] -> [B, C, L] -> [B, L, C]
        embed_t = embed.transpose(1, 2)
        conv_out = torch.cat([F.relu(self.conv3(embed_t)), F.relu(self.conv5(embed_t))], dim=1)
        cnn_out = conv_out.transpose(1, 2)  # [B, L, C]
        cnn_out = self.cnn_norm(cnn_out)
        cnn_out = self.cnn_dropout(cnn_out)
        
        # BiLSTM with residual
        packed = pack_padded_sequence(cnn_out, lengths.cpu(), batch_first=True, enforce_sorted=False)
        out, _ = self.lstm_layers[0](packed)
        out, _ = pad_packed_sequence(out, batch_first=True, total_length=L)
        out = self.lstm_norms[0](out)
        out = self.lstm_dropout(out)
        
        for i in range(1, len(self.lstm_layers)):
            residual = out
            packed = pack_padded_sequence(out, lengths.cpu(), batch_first=True, enforce_sorted=False)
            out, _ = self.lstm_layers[i](packed)
            out, _ = pad_packed_sequence(out, batch_first=True, total_length=L)
            out = self.lstm_norms[i](out)
            out = out + residual  # Residual
            out = self.lstm_dropout(out)
        
        # Attention
        out = self.attention(out, mask)
        
        # Emissions
        q8_emissions = self.q8_emission(out)  # [B, L, num_q8]
        q3_emissions = self.q3_emission(out)  # [B, L, num_q3]
        
        if q8_tags is not None and q3_tags is not None:
            # Training: return negative log-likelihood
            loss_q8 = -self.crf_q8(q8_emissions, q8_tags, mask=mask, reduction='mean')
            loss_q3 = -self.crf_q3(q3_emissions, q3_tags, mask=mask, reduction='mean')
            return loss_q8, loss_q3
        else:
            # Inference: decode best sequence
            q8_preds = self.crf_q8.decode(q8_emissions, mask=mask)
            q3_preds = self.crf_q3.decode(q3_emissions, mask=mask)
            return q8_preds, q3_preds

model = CNN_BiLSTM_CRF(
    vocab_size=len(AA_TO_ID),
    embed_dim=cfg.embed_dim,
    cnn_channels=cfg.cnn_channels,
    hidden_dim=cfg.hidden_dim,
    num_layers=cfg.num_layers,
    num_q8=len(Q8_TO_ID),
    num_q3=len(Q3_TO_ID),
    dropout=cfg.dropout
).to(cfg.device)

print(model)
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")

## Training

In [ ]:
optimizer = optim.AdamW(model.parameters(), lr=cfg.learning_rate, weight_decay=0.01)
scheduler = optim.lr_scheduler.OneCycleLR(optimizer, max_lr=cfg.learning_rate, epochs=cfg.epochs, steps_per_epoch=len(train_loader))

def calc_f1(preds_list, targets, lengths):
    # preds_list: list of lists (from CRF decode)
    all_preds = []
    all_targets = []
    for i, length in enumerate(lengths):
        all_preds.extend(preds_list[i][:length])
        all_targets.extend(targets[i, :length].cpu().numpy().tolist())
    # Remove padding (0)
    filtered = [(p, t) for p, t in zip(all_preds, all_targets) if t != PAD_TOKEN]
    if not filtered:
        return 0.0
    p, t = zip(*filtered)
    return f1_score(t, p, average='macro', zero_division=0)

def train_epoch(model, loader, optimizer, scheduler, device, epoch):
    model.train()
    total_loss, count = 0, 0
    pbar = tqdm(loader, desc=f"Epoch {epoch+1}")
    for batch in pbar:
        seq = batch['seq'].to(device)
        lengths = batch['lengths']
        q8_tgt = batch['q8'].to(device)
        q3_tgt = batch['q3'].to(device)
        
        optimizer.zero_grad()
        loss_q8, loss_q3 = model(seq, lengths, q8_tgt, q3_tgt)
        loss = loss_q8 + loss_q3
        
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        
        total_loss += loss.item()
        count += 1
        pbar.set_postfix({'loss': loss.item(), 'lr': scheduler.get_last_lr()[0]:.6f})
        
        if trk:
            trk.log({'train/loss': loss.item()})
    
    return total_loss / count

def evaluate(model, loader, device):
    model.eval()
    all_q8_preds, all_q3_preds = [], []
    all_q8_tgts, all_q3_tgts = [], []
    all_lengths = []
    total_loss = 0
    count = 0
    
    with torch.no_grad():
        for batch in loader:
            seq = batch['seq'].to(device)
            lengths = batch['lengths']
            q8_tgt = batch['q8'].to(device)
            q3_tgt = batch['q3'].to(device)
            
            # Get loss
            loss_q8, loss_q3 = model(seq, lengths, q8_tgt, q3_tgt)
            total_loss += (loss_q8 + loss_q3).item()
            count += 1
            
            # Get predictions
            q8_preds, q3_preds = model(seq, lengths)
            all_q8_preds.extend(q8_preds)
            all_q3_preds.extend(q3_preds)
            all_q8_tgts.append(q8_tgt)
            all_q3_tgts.append(q3_tgt)
            all_lengths.extend(lengths.tolist())
    
    # Flatten for F1
    q8_tgts_cat = torch.cat(all_q8_tgts, dim=0)
    q3_tgts_cat = torch.cat(all_q3_tgts, dim=0)
    
    f1_q8 = calc_f1(all_q8_preds, q8_tgts_cat, all_lengths)
    f1_q3 = calc_f1(all_q3_preds, q3_tgts_cat, all_lengths)
    
    return total_loss / count, f1_q8, f1_q3

best_score = 0.0
for epoch in range(cfg.epochs):
    train_loss = train_epoch(model, train_loader, optimizer, scheduler, cfg.device, epoch)
    val_loss, val_f1_8, val_f1_3 = evaluate(model, val_loader, cfg.device)
    
    val_score = (2 * val_f1_8 * val_f1_3) / (val_f1_8 + val_f1_3 + 1e-8)
    
    print(f"Epoch {epoch+1}: Train Loss={train_loss:.4f} | Val Score={val_score:.4f} (Q8={val_f1_8:.4f}, Q3={val_f1_3:.4f})")
    
    if trk:
        trk.log({'val/score': val_score, 'val/loss': val_loss})
    
    if val_score > best_score:
        best_score = val_score
        torch.save(model.state_dict(), 'best_model_v5.pth')
        print("  --> Best model saved")

print(f"\nTraining Complete. Best Val Score: {best_score:.4f}")

## Inference

In [ ]:
test_ds = ProteinDataset(cfg.test_path, mode='test')
test_loader = DataLoader(test_ds, batch_size=cfg.batch_size, shuffle=False, collate_fn=collate_fn)

model.load_state_dict(torch.load('best_model_v5.pth'))
model.eval()
print("Loaded best model.")

results = []
with torch.no_grad():
    for batch in tqdm(test_loader, desc="Inference"):
        seq = batch['seq'].to(cfg.device)
        lengths = batch['lengths']
        ids = batch['ids']
        
        q8_preds, q3_preds = model(seq, lengths)
        
        for i, id_ in enumerate(ids):
            length = lengths[i].item()
            q8_str = ''.join([ID_TO_Q8[idx] for idx in q8_preds[i][:length]])
            q3_str = ''.join([ID_TO_Q3[idx] for idx in q3_preds[i][:length]])
            results.append({'id': id_, 'sst8': q8_str, 'sst3': q3_str})

df_sub = pd.DataFrame(results)
df_sub.to_csv('submission_v5.csv', index=False)
print("Saved submission_v5.csv")
df_sub.head()